# Hyperparameter Tuning

Runs a Hyperopt (TPE) search over XGBoost hyperparameters for the Home Credit
default-prediction model.
Uses the same features as `02_training_XGB_v1.ipynb` (LightGBM_v3's `selected_features.xlsx`).

**Pipeline overview**
1. Load `model_dataset_final.parquet`
2. Train/validation split (fixed split reused by Notebook 3)
3. Define the search space & objective function
4. Run the Hyperopt search
5. Resolve & save the best hyperparameters
6. Save search history for diagnostics

**Output:**
- `data/Model-Training/HyperParameters/XGB_v1/xgb_resolved_best.json` — best params found by the search
- `data/Model-Training/HyperParameters/XGB_v1/xgb_trials_history.parquet` — full search history
- `data/Model-Training/HyperParameters/XGB_v1/selected_features.json` — this version's feature list, read by the training notebook

> **Note on reproducibility (Bug 17 in the original):** the original notebook ran a
> live Hyperopt search and then silently overwrote the result with a hardcoded dict
> from a previous run — discarding the live search entirely without any indication.
> This notebook keeps **both** paths but makes the active one explicit via the
> `HYPERPARAMS_SOURCE` setting in Section 1, and always prints which source is in effect.

## 1. Imports & Configuration

`HYPERPARAMS_SOURCE` picks where the hyperparameters come from:

- `'search'` (default) — run the full live Hyperopt search and save the best result.
- `'previous'` — keep the previously-saved resolved-best JSON (fast, skips the search).

Set it with `05_model_pipeline`'s `tuning` param (`HPO_HYPERPARAMS_SOURCE`
env var). The training notebook then finds the number of trees and the
classification threshold itself, by cross-validation.

In [ ]:
import os
import gc; 
import json
from datetime import datetime
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score
from hyperopt import fmin, tpe, hp, Trials, space_eval

DATA_ROOT = os.environ.get("HOMECREDIT_DATA_ROOT", r"a:\Dominic\My Projects\Home Credit\data")
SRC_ROOT  = os.environ.get("HOMECREDIT_SRC_DIR", r"a:\Dominic\My Projects\Home Credit\src")
HPO_DIR  = os.path.join(DATA_ROOT, "Model-Training", "HyperParameters", "XGB_v1")

os.makedirs(HPO_DIR, exist_ok=True)

PATH_MODEL_DATASET = os.path.join(DATA_ROOT, "Model-Training", "ModelDataset", "model_dataset_final.parquet")
PATH_PARS_BEST = os.path.join(HPO_DIR, "xgb_resolved_best.json")
# Same feature list as 02_training_XGB_v1.ipynb, which reuses LightGBM_v3's
PATH_SELECTED_VARS = os.path.join(SRC_ROOT, "Training", "04_Model-Development", "LightGBM_v3")
PATH_TRIALS_HISTORY = os.path.join(HPO_DIR, "xgb_trials_history.csv")
PATH_FEATURE_LIST = os.path.join(HPO_DIR, "selected_features.json")  # read by the training notebook

# ── Reproducibility / split config — MUST match training exactly ────────
RANDOM_STATE = 1994
TEST_SIZE    = 0.20

# ── Where the hyperparameters come from ───────────────────────────────────
#   'search'   -> run the live Hyperopt search below and save its result (default)
#   'previous' -> keep the previously-saved resolved-best JSON (fast, skips the search)
# Set per run by 05_model_pipeline's `tuning` param (HPO_HYPERPARAMS_SOURCE env
# var) without editing the notebook; HPO_USE_PREVIOUS_BEST=true still maps to 'previous'.
# (Training with the library's default hyperparameters is a switch in the training
# notebook instead.)
HYPERPARAMS_SOURCE = os.environ.get("HPO_HYPERPARAMS_SOURCE", "").strip().lower()
if not HYPERPARAMS_SOURCE:
    HYPERPARAMS_SOURCE = ("previous" if os.environ.get("HPO_USE_PREVIOUS_BEST", "False").strip().lower()
                          in ("1", "true", "yes") else "search")
assert HYPERPARAMS_SOURCE in ("search", "previous"), f"unknown HYPERPARAMS_SOURCE {HYPERPARAMS_SOURCE!r}"
# pick a specific number of points from search space, try each one, and give me the best one you found
MAX_EVALS = 150

In [ ]:
print(f"Hyperparameter source: {HYPERPARAMS_SOURCE}")

## 2. Load Data & Split

> **Fix (Bug 7 in the original):** the original used an 80/20 split here for HPO, but
> a *different* 85/15 split later for final training/evaluation in the training
> notebook — meaning the model selected via HPO was never actually validated on the
> exact data split used for final reporting. `RANDOM_STATE` and `TEST_SIZE` are now
> defined once above and must be reused identically in Notebook 3.

In [ ]:
model_dataset = pd.read_parquet(PATH_MODEL_DATASET)
# This version's feature selection rule -- saved to selected_features.json (Section 6)
# and read by 02_training_XGB_v1.ipynb.
features_df = pd.read_excel(os.path.join(PATH_SELECTED_VARS, 'selected_features.xlsx'))
feature_cols = features_df['feature'].tolist()
print(f"Selected {len(feature_cols)} features (same as 02_training_XGB_v1.ipynb (LightGBM_v3's list))")

# Re-derive cat_cols and re-apply 'category' dtype — parquet round-trips pandas
# categories correctly, but we re-assert here defensively in case the file was
# regenerated by a different process.
cat_cols = model_dataset.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    model_dataset[col] = model_dataset[col].astype('category')

X = model_dataset[['SK_ID_CURR'] + feature_cols]
y = model_dataset['TARGET']
X.set_index('SK_ID_CURR', inplace=True)

print(f"X shape: {X.shape}")
print(f"Categorical columns: {len(cat_cols)}")

FEATURE_RULE = "fixed list: 04_Model-Development/LightGBM_v3/selected_features.xlsx"   # recorded with the saved feature list

Convert `float64` type columns to `float32`

In [ ]:
# ADD THIS TO SHRINK THE DATAFRAME IN MEMORY:
float_cols = X.select_dtypes(include=['float64']).columns
X[float_cols] = X[float_cols].astype('float32')

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"X_train: {X_train.shape}  |  X_val: {X_val.shape}")

In [ ]:
del X, y, model_dataset
gc.collect()

## 3. Define Search Space

XGBoost with the `hist` tree method and native categorical support, matching
`02_training_XGB_v1.ipynb`.

> **Note:** `n_estimators` is not part of the search space. It is given a fixed
> ceiling and controlled through early stopping.

In [ ]:
space = {
    # Tree structure
    'max_depth': hp.quniform('max_depth', 3, 10, 1),
    'min_child_weight': hp.loguniform('min_child_weight', np.log(1.0), np.log(200.0)),

    # Learning rate
    'learning_rate': hp.loguniform('learning_rate', np.log(0.01), np.log(0.2)),

    # Row and column sampling
    'subsample': hp.uniform('subsample', 0.5, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.2, 1.0),

    # Regularization
    'reg_alpha':  hp.loguniform('reg_alpha', np.log(1e-3), np.log(100.0)),
    'reg_lambda': hp.loguniform('reg_lambda', np.log(1e-3), np.log(100.0)),
    # Minimum loss reduction required to make a further split
    'gamma': hp.loguniform('gamma', np.log(1e-3), np.log(10.0)),
}

INT_PARAMS = ['max_depth']

## 4. Objective Function

In [ ]:
def objective(params):
    """
    Hyperopt objective: 5-fold CV mean AUC for one XGBoost hyperparameter set.

    Hyperopt minimizes the return value, so we negate AUC (higher AUC = lower loss).
    Manual K-fold so each fold can use XGBoost's native early stopping. Capped at 500
    trees with max_bin=63 for speed -- Section 7 re-runs CV with the exact training model.
    """
    current_params = params.copy()
    for p in INT_PARAMS:
        current_params[p] = int(current_params[p])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    auc_scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]

        model = xgb.XGBClassifier(
            objective='binary:logistic',
            eval_metric='auc',
            tree_method='hist',
            enable_categorical=True,
            max_bin=63,
            random_state=42,
            n_jobs=14,  # leave cores available for Airflow
            n_estimators=500,
            early_stopping_rounds=20,
            **current_params,
        )
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

        preds = model.predict_proba(X_va)[:, 1]
        auc_scores.append(roc_auc_score(y_va, preds))
        del X_tr, y_tr, X_va, y_va, model
        gc.collect()

    mean_auc = np.mean(auc_scores)
    return {'loss': -mean_auc, 'status': 'ok'}

## 5. Run / Load Search

Controlled by `HYPERPARAMS_SOURCE` in Section 1. Each mode produces a single,
clearly-labelled `resolved_best` dict and prints which source produced it — there is
no silent overwrite as in the original notebook.

In [ ]:
if HYPERPARAMS_SOURCE == 'previous':
    print(f"HYPERPARAMS_SOURCE='previous' → loading saved params from {PATH_PARS_BEST}")
    with open(PATH_PARS_BEST) as f:
        resolved_best = json.load(f)
    trials = None

else:
    print(f"HYPERPARAMS_SOURCE='search' → running live Hyperopt search ({MAX_EVALS} evals)")
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=MAX_EVALS,
        trials=trials,
    )
    print("\n--- Hyperopt Search Complete ---")

    resolved_best = space_eval(space, best)
    for p in INT_PARAMS:
        resolved_best[p] = int(resolved_best[p])

print(f"\nActive hyperparameter source: {HYPERPARAMS_SOURCE}")
print(json.dumps(resolved_best, indent=4) if resolved_best else "(none)")

## 6. Save Results

Writes `resolved_best` to disk only after a live search (`'previous'` leaves the
saved file untouched), along with the full trial history.

> **Fix (Bug 16 in the original):** `trials_df` / `trials_history` were computed but
> never saved — discarding useful diagnostic data about the search trajectory. Now
> saved to `lgbm_trials_history.parquet` whenever a live search runs.

In [ ]:
if HYPERPARAMS_SOURCE == 'search':   # 'previous' leaves the saved file as is
    with open(PATH_PARS_BEST, 'w') as f:
        json.dump(resolved_best, f, indent=4)
        print(f"Saved resolved_best → {PATH_PARS_BEST}")

if trials is not None:
    trials_df = pd.DataFrame(trials.results)
    trials_df['auc'] = -trials_df['loss']
    trials_df['trial_number'] = range(len(trials_df))

    params_df = pd.DataFrame([t['misc']['vals'] for t in trials.trials])
    params_df = params_df.applymap(lambda x: x[0] if isinstance(x, list) and x else None)

    trials_history = pd.concat([trials_df, params_df], axis=1)
    trials_history.to_csv(PATH_TRIALS_HISTORY, index=False)
    print(f"Saved trial history ({len(trials_history)} trials) → {PATH_TRIALS_HISTORY}")

    print(f"\nBest CV AUC found: {trials_history['auc'].max():.4f}")
else:
    print("No live search ran — trial history not available this run.")

# Feature list for the training notebook, saved on every run (search or previous): training
# reads this file instead of selecting features itself, so both always use the same list
feature_list = {
    'features': list(feature_cols),
    'rule': FEATURE_RULE,
    'hyperparameters_source': HYPERPARAMS_SOURCE,
    'created_at': datetime.now().isoformat(timespec='seconds'),
}
with open(PATH_FEATURE_LIST, 'w') as f:
    json.dump(feature_list, f, indent=2)
print(f"Saved feature list ({len(feature_cols)} features; {FEATURE_RULE}) → {PATH_FEATURE_LIST}")

## 7. Cleanup

In [ ]:
del X_train, X_val, y_train, y_val
gc.collect()
print("All DataFrames released.")